In [1]:
import pandas as pd

ratings = pd.read_csv(
    "C:\\surya\\ml\\Movie-Recommendation-System\\data\\raw\\ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"]
)

ratings.head()

,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [2]:
from surprise import Dataset
from surprise import Reader

reader = Reader(
    rating_scale=(1, 5)
)

data = Dataset.load_from_df(
    ratings[["userId", "movieId", "rating"]],
    reader
)

In [3]:
from surprise.model_selection import train_test_split

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

In [4]:
from surprise import SVD

model = SVD()

model.fit(trainset)

In [5]:
from surprise import accuracy

predictions = model.test(testset)

accuracy.rmse(predictions)

RMSE: 0.8729


np.float64(0.872888994918327)

In [6]:
accuracy.rmse(predictions)

RMSE: 0.8729


np.float64(0.872888994918327)

In [7]:
all_movies = ratings["movieId"].unique()

user_id = 1

watched_movies = ratings[
    ratings["userId"] == user_id
]["movieId"].values

In [8]:
unwatched_movies = [
    movie
    for movie in all_movies
    if movie not in watched_movies
]

In [9]:
predictions = []

for movie_id in unwatched_movies:

    pred = model.predict(
        user_id,
        movie_id
    )

    predictions.append(
        (movie_id, pred.est)
    )

In [10]:
predictions = sorted(
    predictions,
    key=lambda x: x[1],
    reverse=True
)

In [11]:
top_10 = predictions[:10]

top_10

[(np.int64(2905), np.float64(4.923796554617257)),
 (np.int64(593), np.float64(4.898114484841313)),
 (np.int64(590), np.float64(4.851774484420607)),
 (np.int64(110), np.float64(4.840836303787869)),
 (np.int64(670), np.float64(4.8170118838336755)),
 (np.int64(1250), np.float64(4.7709314772586)),
 (np.int64(3338), np.float64(4.760416589656212)),
 (np.int64(318), np.float64(4.745241634343758)),
 (np.int64(1704), np.float64(4.7330220242403245)),
 (np.int64(1361), np.float64(4.725414345163636))]

In [15]:
movies = pd.read_csv(
    "C:\\surya\\ml\\Movie-Recommendation-System\\data\\raw\\movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["movieId", "title", "genres"]
)

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [16]:
recommended_movies = pd.DataFrame(
    top_10,
    columns=["movieId", "predicted_rating"]
)

recommended_movies = recommended_movies.merge(
    movies,
    on="movieId"
)

recommended_movies[
    ["title", "predicted_rating"]
]

,title,predicted_rating
0,Sanjuro (1962),4.923797
1,"Silence of the Lambs, The (1991)",4.898114
2,Dances with Wolves (1990),4.851774
3,Braveheart (1995),4.840836
4,"World of Apu, The (Apur Sansar) (1959)",4.817012
5,"Bridge on the River Kwai, The (1957)",4.770931
6,For All Mankind (1989),4.760417
7,"Shawshank Redemption, The (1994)",4.745242
8,Good Will Hunting (1997),4.733022
9,Paradise Lost: The Child Murders at Robin Hood...,4.725414


In [17]:
print(movies.shape)

(3883, 3)
